MANC notebook- create csv annotation fiels for MANC in version 1.2.3. 2026 04 03

In [3]:
# set up environment
import sys
import json
import re
import requests
import pandas as pd
from pathlib import Path
import json
import csv
from caveclient import CAVEclient

import urllib.parse


print("Python executable:", sys.executable)
print("Imports OK")

Python executable: c:\Users\JHS\Documents\Python\cave_env\Scripts\python.exe
Imports OK


# set up in and output directories and state names 

In [5]:
# set up directories
PROJECT_ROOT = Path.cwd()

INPUT_DIR = PROJECT_ROOT / "output"
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input directory missing: {INPUT_DIR}")

OUTPUT_DIR = PROJECT_ROOT / "output-MANCv1.2.3_04B_morphology_clusters_T1"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Input:", INPUT_DIR)
print("Output:", OUTPUT_DIR)

# load input state
STATE_IN = INPUT_DIR / "manc_v1p2_with_manual_IR_layers.json"
if not STATE_IN.exists():
    raise FileNotFoundError(f"Missing state file: {STATE_IN}")

with open(STATE_IN, "r", encoding="utf-8") as f:
    state = json.load(f)

# save output state
state_title = state.get("title", "untitled_state")
STATE_OUT = OUTPUT_DIR / "manc_4b_morphology_clusters_jhs.json"

with open(STATE_OUT, "w", encoding="utf-8") as f:
    json.dump(state, f, indent=2)

Input: c:\Users\JHS\Documents\Python\project_folder_4B\output
Output: c:\Users\JHS\Documents\Python\project_folder_4B\output-MANCv1.2.3_04B_morphology_clusters_T1


# start with the segmentation done in olv MANC version
## build a simple descriptive CSV to describe the input state view

In [6]:
# build a simple descriptive CSV to describe the state view

# ---- LOAD INPUT STATE (read only) ----
with open(STATE_IN, "r", encoding="utf-8") as f:
    state = json.load(f)

# ---- EXTRACT LAYERS ----
rows = []

for L in state.get("layers", []):
    layer_type = L.get("type", "")
    name = L.get("name", "")
    visible = L.get("visible", "")

    segments = L.get("segments", [])
    segments_IDs = ",".join(str(s) for s in segments) if segments else ""

    seg_colors = L.get("segmentColors") or {}
    unique_colors = set(seg_colors.values())
    layer_color = list(unique_colors)[0] if len(unique_colors) == 1 else ""

    rows.append({
        "state_title": state_title,
        "layer_name": name,
        "layer_type": layer_type,
        "visible": visible,
        "layer_color": layer_color,
        "num_segments": len(segments),
        "segments_IDs": segments_IDs,
        "note": ""
    })

# ---- WRITE CSV ----
fieldnames = [
    "state_title",
    "layer_name",
    "layer_type",
    "visible",
    "layer_color",
    "num_segments",
    "segments_IDs",
    "note",
]

# define output filename based on input state
OUT_CSV = OUTPUT_DIR / f"{STATE_IN.stem}_layer_summary.csv"

# write CSV
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote summary of {STATE_IN.name} → {OUT_CSV.name}")

Wrote summary of manc_v1p2_with_manual_IR_layers.json → manc_v1p2_with_manual_IR_layers_layer_summary.csv


### report the segmentation I have done, but with a row for each neuron, and add all anntoations for that neuron as column

In [10]:
# All neurons: build a CSV mapping segment IDs to manual morphology clusters

# ---- LOAD INPUT STATE ----
with open(STATE_IN, "r", encoding="utf-8") as f:
    state = json.load(f)

# ---- DEFINE OUTPUT ----
OUT_CSV = OUTPUT_DIR / f"{STATE_IN.stem}_manual_clusters_by_segment.csv"

rows = []

state_title = state.get("title", "untitled_state")

# ---- EXTRACT PER-SEGMENT DATA ----
for L in state.get("layers", []):
    layer_name = L.get("name", "")
    layer_type = L.get("type", "")
    visible = L.get("visible", "")

    segments = L.get("segments", [])

    for seg_id in segments:
        rows.append({
            "state_title": state_title,
            "segment_id": str(seg_id),
            "manual_morphology_cluster": layer_name,
            "manc_type": "",  # to be filled later
            "layer_type": layer_type,
            "visible": visible,
            "note": ""
        })

# ---- WRITE CSV ----
fieldnames = [
    "state_title",
    "segment_id",
    "manual_morphology_cluster",
    "manc_type",
    "layer_type",
    "visible",
    "note",
]

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote segment → cluster mapping: {OUT_CSV.name}")

Wrote segment → cluster mapping: manc_v1p2_with_manual_IR_layers_manual_clusters_by_segment.csv


In [11]:

# Build one master DataFrame with one row per segment ID

# ---- LOAD INPUT STATE ----
with open(STATE_IN, "r", encoding="utf-8") as f:
    state = json.load(f)

rows = []

for L in state.get("layers", []):
    layer_name = L.get("name", "")
    layer_type = L.get("type", "")
    visible = L.get("visible", "")
    segment_ids = L.get("segments", []) or []

    if layer_type != "segmentation":
        continue
    if not segment_ids:
        continue

    for seg_id in segment_ids:
        rows.append({
            "manual_morphology_cluster": layer_name,
            "layer_type": layer_type,
            "visible": visible,
            "segment_id": int(seg_id),
            "manc_type": "",   # fill later
            "note": ""
        })

all_layers_df = (
    pd.DataFrame(rows)
    .drop_duplicates()
    .sort_values(["manual_morphology_cluster", "segment_id"])
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

all_layers_df

,manual_morphology_cluster,layer_type,visible,segment_id,manc_type,note
0,all-synaptic,segmentation,,1,,
1,all-tissue,segmentation,False,1,,
2,court et al. tracts,segmentation,,100,,
3,court et al. tracts,segmentation,,101,,
4,court et al. tracts,segmentation,,102,,
5,court et al. tracts,segmentation,,103,,
6,court et al. tracts,segmentation,,104,,
7,court et al. tracts,segmentation,,105,,
8,court et al. tracts,segmentation,,106,,
9,manc-seg-v1.2 4B manual IR- proprioceptice ta...,segmentation,False,21862,,


In [ ]:
#inspect layer/ morphology cluster names
for name in sorted(all_layers_df["manual_morphology_cluster"].unique()):
    print(name)

all-synaptic
all-tissue
court et al. tracts
manc-seg-v1.2 4B manual  IR- proprioceptice tactile n=3
manc-seg-v1.2 4B manual  IR-nomorphology assigned-early secondary
manc-seg-v1.2 4B manual BR-1 n=6
manc-seg-v1.2 4B manual BR-2 n=10
manc-seg-v1.2 4B manual BR-3 n=4
manc-seg-v1.2 4B manual IR nomorphology assigned secondary
manc-seg-v1.2 4B manual IR-1
manc-seg-v1.2 4B manual IR-2
manc-seg-v1.2 4B manual IR-3
manc-seg-v1.2 4B manual IR-4
manc-seg-v1.2 4B manual IR-independent leg n=11
manc-seg-v1.2 4B- primary  n=4
manc-seg-v1.2 4b-manual BR-4 n=6
manc-seg-v1.2-4B-all
manc-seg-v1.2-4B-subbclass BI n=4
manc-seg-v1.2-4B-sublass IR n=60
manc-seg-v1.21- secondary n=73
manc-seg-v1.21-4B subclass BA n=1
manc-seg-v1.21-4B-subclassBR n=26
manc-seg-v1.22-4B subclass IA=4
manc-seg-v1.22-4B-early secondary n= 20
manc:v1.2.1-TTMn
nerves
neuropils


In [ ]:
LAYERS_TO_ANALYZE = [
    "manc-seg-v1.2 4B manual  IR- proprioceptice tactile n=3",
    "manc-seg-v1.2 4B manual BR-1 n=6",
    "manc-seg-v1.2 4B manual BR-2 n=10",
    "manc-seg-v1.2 4B manual BR-3 n=4",
    "manc-seg-v1.2 4B manual IR-1",
    "manc-seg-v1.2 4B manual IR-2",
    "manc-seg-v1.2 4B manual IR-3",
    "manc-seg-v1.2 4B manual IR-4",
    "manc-seg-v1.2 4B manual IR-independent leg n=11",
    "manc-seg-v1.2 4b-manual BR-4 n=6",
    "manc-seg-v1.2-4B-subbclass BI n=4",
    "manc-seg-v1.2-4B-sublass IR n=60",
    "manc-seg-v1.21-4B subclass BA n=1",
    "manc-seg-v1.21-4B-subclassBR n=26",
    "manc-seg-v1.22-4B subclass IA=4",
]

selected_layers_df = (
    all_layers_df[
        all_layers_df["manual_morphology_cluster"].isin(LAYERS_TO_ANALYZE)
    ]
    .copy()
    .sort_values(["manual_morphology_cluster", "segment_id"])
    .reset_index(drop=True)
)

selected_layers_df

In [23]:
# check if this ID list is compelte and does nto have duplicates

#is the same neuron added to multipel morphology clusters?
dupes = selected_layers_df[
    selected_layers_df.duplicated(subset="segment_id", keep=False)
].sort_values("segment_id")

print("Total duplicated segment IDs:", dupes["segment_id"].nunique())
dupes


#is it complete?

# ---- define the reference layer: 4B all ----
REFERENCE_LAYER = "manc-seg-v1.2-4B-all"   # replace with exact layer name from your state

# ---- get all IDs from the reference layer ----
layer_4b_all_df = (
    all_layers_df[
        all_layers_df["manual_morphology_cluster"] == REFERENCE_LAYER
    ]
    .copy()
    .sort_values("segment_id")
    .reset_index(drop=True)
)

layer_4b_all_ids = set(layer_4b_all_df["segment_id"])

# ---- get all IDs from your selected layers ----
selected_ids = set(selected_layers_df["segment_id"])

# ---- compare coverage ----
missing_from_selection = sorted(layer_4b_all_ids - selected_ids)
extra_in_selection = sorted(selected_ids - layer_4b_all_ids)

print("IDs in 4B all:", len(layer_4b_all_ids))
print("IDs in selected layers:", len(selected_ids))
print("Missing from selection:", len(missing_from_selection))
print("Extra in selection:", len(extra_in_selection))

Total duplicated segment IDs: 10
IDs in 4B all: 97
IDs in selected layers: 76
Missing from selection: 21
Extra in selection: 0


# Fix these errors when I review the updated version of MANC 
## build the T1 morphological clsuters up for the latest MANC version, v 1.2.3

In [16]:
#----Imports
import os
import json
import pandas as pd
from pathlib import Path
#pip install neuprint-python
from neuprint import Client, fetch_neurons, fetch_custom, NeuronCriteria as NC


In [20]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env from current working directory

token = os.environ["NEUPRINT_APPLICATION_CREDENTIALS"]

In [ ]:
from neuprint import Client

c = Client(
    "https://neuprint.janelia.org",
    dataset="manc:v1.2.3",
    token=token
)
print(c)

Client("https://neuprint.janelia.org", "manc:v1.2.3")


In [24]:
#Fetch neurons to popualte the state later: one neuromere with 04B 

from neuprint import fetch_custom

cypher = """
MATCH (n:Neuron)
WHERE n.hemilineage = '04B'
  AND n.somaNeuromere = 'T1'
  AND n.somaSide = 'LHS'
RETURN
  n.bodyId AS bodyId,
  n.type AS type,
  n.instance AS instance,
  n.class AS class,
  n.subclass AS subclass,
  n.hemilineage AS hemilineage,
  n.somaNeuromere AS somaNeuromere,
  n.somaSide AS somaSide,
  n.rootSide AS rootSide,
  n.longTract AS longTract,
  n.birthtime AS birthtime
ORDER BY bodyId
"""

df_4b_t1_lhs = fetch_custom(cypher, client=c)
#df_4b_t1_lhs.head() 
#df_4b_t1_lhs["bodyId"].nunique() # n=97

In [29]:
# Generate a new state that diplays the 4B morphology clusters, as layers for each subclass WITH  all neurons in one layer displayed with the same color

import json
import urllib.parse
from itertools import cycle


#generate new, clean state:

##define sources:
seg_source = None
em_source = None

for L in state["layers"]:
    layer_type = L.get("type")
    src = L.get("source")

    if layer_type == "segmentation" and seg_source is None:
        seg_source = src

    if layer_type == "image" and em_source is None:
        em_source = src

assert seg_source is not None, "Segmentation source not found"
assert em_source is not None, "EM source not found"

#generate state
new_state = {
    "title": "MANC_v1.2.3_4B_morphology_clusters_JHS",
    "layout": "3d",
    "showSlices": True,
    "projectionBackgroundColor": "#ffffff",
    "crossSectionBackgroundColor": "#ffffff",  
    "showLayerPanel": True, 
    "layers": [
        {
            "type": "image",
            "name": "EM",
            "source": em_source,
            "visible": True
        }
    ]
}


# --- Color palette (cycles if needed) ---
COLORS = [
    "#87CEEB",  # sky blue
    "#FFA500",  # orange
    "#E34234",  # vermillion
    "#CC99FF",  # pale violet
    "#66CDAA",  # medium aquamarine
    "#FFD700",  # gold
]

color_cycle = cycle(COLORS)


# --- Helper: upsert layer (FIXED) ---
def add_segmentation_layer(state, name, body_ids, seg_source, color):
    body_ids = sorted(set(int(x) for x in body_ids))

    layer = {
        "type": "segmentation",
        "name": name,
        "source": seg_source,
        "segments": [str(x) for x in body_ids],
        "segmentColors": {str(x): color for x in body_ids},
        "visible": True,
    }

    # replace if exists
    for i, existing in enumerate(state["layers"]):
        if existing.get("name") == name:
            state["layers"][i] = layer
            return

    # otherwise append
    state["layers"].append(layer)


# --- Clean subclass labels ---
subclass_df = df_4b_t1_lhs.copy()
subclass_df["subclass"] = (
    subclass_df["subclass"]
    .fillna("unclassified")
    .astype(str)
    .str.strip()
)
subclass_df.loc[subclass_df["subclass"] == "", "subclass"] = "unclassified"


# --- Inspect counts ---
subclass_counts = (
    subclass_df.groupby("subclass")["bodyId"]
    .nunique()
    .sort_values(ascending=False)
)
print(subclass_counts)


# --- Build layers with colors ---
for subclass_name, group in subclass_df.groupby("subclass", sort=True):
    body_ids = group["bodyId"].dropna().astype(int).tolist()
    if not body_ids:
        continue

    layer_name = f"MANC_v1.2.3_{subclass_name}"
    color = next(color_cycle)

    add_segmentation_layer(new_state, layer_name, body_ids, seg_source, color)


# --- Debug print ---
print("State title:", new_state["title"])
print("Number of layers:", len(new_state["layers"]))

for layer in new_state["layers"]:
    print(layer["name"], len(layer.get("segments", [])))


# --- Save ---
OUTPUT_DIR.mkdir(exist_ok=True)

STATE_OUT = OUTPUT_DIR / "MANC_v1.2.3_4B_morphology_clusters_JHS.json"

with open(STATE_OUT, "w", encoding="utf-8") as f:
    json.dump(new_state, f, indent=2)


# --- Generate Neuroglancer URL ---
encoded = urllib.parse.quote(json.dumps(new_state, separators=(",", ":")))
NEW_URL = "https://clio-ng.janelia.org/#!" + encoded

print("Saved new state:", STATE_OUT)
print("\nNew URL:\n")
print(NEW_URL)

subclass
IR    60
BR    26
BI     6
IA     4
BA     1
Name: bodyId, dtype: int64
State title: MANC_v1.2.3_4B_morphology_clusters_JHS
Number of layers: 6
EM 0
MANC_v1.2.3_BA 1
MANC_v1.2.3_BI 6
MANC_v1.2.3_BR 26
MANC_v1.2.3_IA 4
MANC_v1.2.3_IR 60
Saved new state: c:\Users\JHS\Documents\Python\project_folder_4B\output-MANCv1.2.3_04B_morphology_clusters_T1\MANC_v1.2.3_4B_morphology_clusters_JHS.json

New URL:

https://clio-ng.janelia.org/#!%7B%22title%22%3A%22MANC_v1.2.3_4B_morphology_clusters_JHS%22%2C%22layout%22%3A%223d%22%2C%22showSlices%22%3Atrue%2C%22projectionBackgroundColor%22%3A%22%23ffffff%22%2C%22crossSectionBackgroundColor%22%3A%22%23ffffff%22%2C%22showLayerPanel%22%3Atrue%2C%22layers%22%3A%5B%7B%22type%22%3A%22image%22%2C%22name%22%3A%22EM%22%2C%22source%22%3A%7B%22url%22%3A%22precomputed%3A//gs%3A//flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg%22%2C%22subsources%22%3A%7B%22default%22%3Atrue%7D%2C%22enableDefaultSubsources%22%3Afalse%7D%2C%22visible%2

In [35]:
#inspect MANC morphology annotations etc 

print(df_4b_t1_lhs.columns)

type_counts = (
    df_4b_t1_lhs.groupby("type")["bodyId"]
    .nunique()
    .sort_values(ascending=False)
)

print(type_counts.head(20))


typed_total = type_counts.sum()
print("Total typed neurons:", typed_total)


table = (
    df_4b_t1_lhs
    .groupby(["class","subclass", "type", "instance"])["bodyId"]
    .nunique()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print(table)

Index(['bodyId', 'type', 'instance', 'class', 'subclass', 'hemilineage',
       'somaNeuromere', 'somaSide', 'rootSide', 'longTract', 'birthtime'],
      dtype='object')
type
IN04B013    4
IN04B081    4
IN04B101    4
IN04B100    4
IN04B015    3
IN04B009    3
IN04B079    3
IN04B010    3
IN04B091    3
IN04B094    3
IN04B034    2
IN04B024    2
IN04B073    2
IN04B078    2
IN04B041    2
IN04B038    2
IN04B067    2
IN04B066    2
IN04B093    2
IN04B070    2
Name: bodyId, dtype: int64
Total typed neurons: 97
               class subclass      type       instance  count
11  intrinsic neuron       BR  IN04B013  IN04B013_T1_L      4
41  intrinsic neuron       IR  IN04B081  IN04B081_T1_L      4
50  intrinsic neuron       IR  IN04B100  IN04B100_T1_L      4
21  intrinsic neuron       BR  IN04B079  IN04B079_T1_L      3
51  intrinsic neuron       IR  IN04B101  IN04B101_T1_L      3
27  intrinsic neuron       IR  IN04B015  IN04B015_T1_L      3
25  intrinsic neuron       IR  IN04B009  IN04B009_T1_L      

In [41]:
#fidn the annotations the way I see them when I lelec a neuron inthe API

from neuprint import fetch_neurons, NeuronCriteria as NC

criteria = NC(bodyId=df_4b_t1_lhs["bodyId"].tolist())

neurons, roi_counts = fetch_neurons(criteria, client=c)

print(neurons.columns)

Index(['bodyId', 'instance', 'type', 'pre', 'post', 'downstream', 'upstream',
       'size', 'status', 'statusLabel', 'somaLocation', 'roiInfo',
       'celltypeTotalNtPredictions', 'ntUnknownProb', 'subclass',
       'transmission', 'rootSide', 'somaNeuromere', 'vfbId', 'prefix', 'group',
       'systematicType', 'serialMotif', 'somaSide', 'receptorType',
       'ntGlutamateProb', 'rootLocation', 'totalNtPredictions',
       'predictedNtProb', 'cluster', 'hemilineage', 'ntAcetylcholineProb',
       'celltypePredictedNt', 'subclassabbr', 'modality', 'longTract',
       'source', 'avgLocation', 'birthtime', 'synweight', 'class',
       'locationType', 'origin', 'location', 'ntGabaProb', 'serial', 'tag',
       'predictedNt', 'synonyms', 'description', 'target', 'exitNerve',
       'entryNerve', 'subcluster', 'tosomaLocation', 'inputRois',
       'outputRois'],
      dtype='object')


In [53]:
#make a descriptive csv, with MANC annotations. Later see how they match mine
# build table
table = (
    neurons_clean
    .groupby(["subclass", "type", "instance", "serialMotif"])["bodyId"]
    .nunique()
    .reset_index(name="count")
)

# add flag AFTER grouping
table["has_motif"] = table["serialMotif"] != "none"

# sort
table = table.sort_values(
    by=["has_motif", "subclass", "count"],
    ascending=[False, True, False]
)

print(table)


   subclass      type       instance      serialMotif  count  has_motif
5        BR  IN04B008  IN04B008_T1_L  independent leg      1       True
21       IA  AN04B001  AN04B001_T1_L          complex      1       True
22       IA  AN04B003  AN04B003_T1_L        ascending      1       True
23       IA  AN04B004  AN04B004_T1_L          complex      1       True
24       IA  AN04B023  AN04B023_T1_L        ascending      1       True
51       IR  IN04B100  IN04B100_T1_L  independent leg      3       True
41       IR  IN04B078  IN04B078_T1_L  independent leg      2       True
45       IR  IN04B092  IN04B092_T1_L  independent leg      2       True
28       IR  IN04B025  IN04B025_T1_L  independent leg      1       True
29       IR  IN04B026  IN04B026_T1_L  independent leg      1       True
30       IR  IN04B031  IN04B031_T1_L  independent leg      1       True
33       IR  IN04B037  IN04B037_T1_L  independent leg      1       True
48       IR  IN04B095  IN04B095_T1_L  independent leg      1    

In [57]:
#check these groupd manually in the viewer: 
table_with_ids = (
    neurons_clean
    .groupby(["subclass", "type", "instance", "serialMotif"])
    .agg(
        count=("bodyId", "nunique"),
        bodyIds=("bodyId", lambda x: sorted(set(x)))
    )
    .reset_index()
)

table_with_ids["has_motif"] = table_with_ids["serialMotif"] != "none"

table_with_ids = table_with_ids.sort_values(
    by=["has_motif", "subclass", "count"],
    ascending=[False, True, False]
)

print(table_with_ids)

   subclass      type       instance      serialMotif  count  \
5        BR  IN04B008  IN04B008_T1_L  independent leg      1   
21       IA  AN04B001  AN04B001_T1_L          complex      1   
22       IA  AN04B003  AN04B003_T1_L        ascending      1   
23       IA  AN04B004  AN04B004_T1_L          complex      1   
24       IA  AN04B023  AN04B023_T1_L        ascending      1   
51       IR  IN04B100  IN04B100_T1_L  independent leg      3   
41       IR  IN04B078  IN04B078_T1_L  independent leg      2   
45       IR  IN04B092  IN04B092_T1_L  independent leg      2   
28       IR  IN04B025  IN04B025_T1_L  independent leg      1   
29       IR  IN04B026  IN04B026_T1_L  independent leg      1   
30       IR  IN04B031  IN04B031_T1_L  independent leg      1   
33       IR  IN04B037  IN04B037_T1_L  independent leg      1   
48       IR  IN04B095  IN04B095_T1_L  independent leg      1   
49       IR  IN04B097  IN04B097_T1_L  independent leg      1   
55       IR  IN04B104  IN04B104_T1_L  in

In [ ]:
#let me inspect subclass IR in 04B to see if MANC has subdivided it 
ir_df = neurons[neurons["subclass"] == "IR"].copy()


ir_df[[
    "bodyId",
    "type",
    "instance",
    "serialMotif",
    "description",
    "modality"
]].sort_values("type")

# create flag
ir_df["maybe_20A"] = ir_df["description"].str.contains(
    "20A", case=False, na=False
)

# subset
ir_20A = ir_df[ir_df["maybe_20A"]]

ir_20A[[
    "bodyId", "type", "instance", "description", 'serialMotif'
]]

# everything else
ir_04B = ir_df[~ir_df["maybe_20A"]]

#look at descriptions for the two IR groups:
df_ids = ir_04B[[
    "bodyId",
    "type",
    "instance",
    "description",
    "serialMotif"
]].drop_duplicates().sort_values("bodyId")

df_ids

# make unique type lists for the two IR groups

types_20A = (
    ir_20A
    .groupby("type")["bodyId"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="bodyIds")
    .sort_values("type")
    .reset_index(drop=True)
)

types_04B_or_unknown = (
    ir_04B
    .groupby("type")["bodyId"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="bodyIds")
    .sort_values("type")
    .reset_index(drop=True)
)

types_20A
types_04B_or_unknown

types_20A_set = set(types_20A["type"])
types_04B_set = set(types_04B_or_unknown["type"])

overlap = types_20A_set & types_04B_set
only_10A = types_20A_set - types_04B_set
only_04B = types_04B_set - types_20A_set

print("Overlap:", overlap)
print("\nOnly in 20A:", only_10A)
print("\nOnly in 04B/unknown:", only_04B)


n_20A = ir_df[ir_df["maybe_20A"]]["bodyId"].nunique()
n_04B = ir_df[~ir_df["maybe_20A"]]["bodyId"].nunique()

print("IR maybe 20A:", n_20A)
print("IR 04B/unknown:", n_04B)

ids_20A = [int(x) for x in ir_20A["bodyId"].unique()]
ids_04B = [int(x) for x in ir_04B["bodyId"].unique()]

print(ids_20A)
print(ids_04B)
df_ids

#100525 is flagged as 9A; mf looks 4B. keep in 4B

Overlap: {'IN04B101', 'IN04B102'}

Only in 20A: {'IN04B081', 'IN04B059', 'IN04B095', 'IN04B092', 'IN04B015', 'IN04B070', 'IN04B112', 'IN04B025', 'IN04B098', 'IN04B026', 'IN04B115', 'IN04B009', 'IN04B097', 'IN04B093'}

Only in 04B/unknown: {'IN04B031', 'IN04B034', 'IN04B104', 'IN04B078', 'IN04B038', 'IN04B014', 'IN04B066', 'IN04B037', 'IN04B100', 'IN04B069', 'IN04B085', 'IN04B091', 'IN04B111', 'IN04B053', 'IN04B094', 'IN04B072'}
IR maybe 20A: 29
IR 04B/unknown: 31
[13128, 13409, 13770, 13798, 13975, 14024, 14697, 14954, 16224, 17567, 18156, 18253, 18629, 18724, 18990, 19833, 19985, 20585, 21189, 21322, 21862, 22868, 23716, 23790, 24181, 26388, 27760, 102536, 152655]
[14389, 15811, 16757, 17572, 17678, 17935, 18605, 18641, 18785, 18831, 20077, 20836, 20932, 21499, 21567, 21808, 22163, 22997, 26941, 29988, 36722, 38071, 100339, 100525, 101453, 102138, 153878, 158391, 164146, 165318, 166421]


,bodyId,type,instance,description,serialMotif
15,14389,IN04B014,IN04B014_T1_L,None,None
25,15811,IN04B034,IN04B034_T1_L,None,None
31,16757,IN04B034,IN04B034_T1_L,4B? in T1 LHS,None
38,17572,IN04B038,IN04B038_T1_L,4B in T1 LHS,None
39,17678,IN04B031,IN04B031_T1_L,None,None
40,17935,IN04B094,IN04B094_T1_L,None,None
45,18605,IN04B031,IN04B031_T1_L,4B? in T1 LHS,independent leg
47,18641,IN04B069,IN04B069_T1_L,None,None
50,18785,IN04B038,IN04B038_T1_L,4B in T1 LHS,None
51,18831,IN04B037,IN04B037_T1_L,4B? in T1 LHS,independent leg


In [67]:
#Jelly manual cluster annotation- reviewed in neuroglancer for morphologies
br_all = [10024, 12523, 13455, 13506, 13535, 13589, 14728, 14774, 15002, 15005, 15309, 16487, 16527, 17351, 17409, 17534, 18148, 18571, 18755, 19250, 19366, 21956, 101447, 152244, 155236, 162540]

br1_ids = [10024, 12523, 15309, 16487, 16527, 17351, 17409, 17534, 18571, 21956, 101447, 152244, 162540]
br2_ids = [13506, 14728, 14774, 15005, 18148, 19250, 19366, 13589, 13455]
br3_ids = [13535, 15002, 18755, 155236]
br_remaining = [x for x in br_all if x not in br1_ids and x not in br2_ids]

print(br_remaining)



DF_JHS_clusters = pd.DataFrame({
    "bodyId": br1_ids,
    "cluster": "BR-1",
    "cluster": "BR-2",
    "cluster": "BR-3"
})

print(DF_JHS_clusters)

print(br_remaining)

# ID 13513 appears to not show up

[13535, 15002, 18755, 155236]
    bodyId cluster
0    10024    BR-3
1    12523    BR-3
2    15309    BR-3
3    16487    BR-3
4    16527    BR-3
5    17351    BR-3
6    17409    BR-3
7    17534    BR-3
8    18571    BR-3
9    21956    BR-3
10  101447    BR-3
11  152244    BR-3
12  162540    BR-3
[13535, 15002, 18755, 155236]


In [ ]:
#Jelly manual cluster annotation- reviewed in neuroglancer for morphologies

ir_all = [13128, 13409, 13770, 13798, 13975, 14024, 14389, 14697, 14954, 15811, 16224, 16757, 17567, 17572, 17678, 17935, 18156, 18253, 18605, 18629, 18641, 18724, 18785, 18831, 18990, 19833, 19985, 20077, 20585, 20836, 20932, 21189, 21322, 21499, 21567, 21808, 21862, 22163, 22868, 22997, 23716, 23790, 24181, 26388, 26941, 27760, 29988, 36722, 38071, 100339, 100525, 101453, 102138, 102536, 152655, 153878, 158391, 164146, 165318, 166421]

ir_04B = [14389, 15811, 16757, 17572, 17678, 17935, 18605, 18641, 18785, 18831, 20077, 20836, 20932, 21499, 21567, 21808, 22163, 22997, 26941, 29988, 36722, 38071, 100339, 100525, 101453, 102138, 153878, 158391, 164146, 165318, 166421]

#Jelly clusters: 2 stemps fro mthe split of IR into 1)20A mancy tag and 2- notaoA (aka 4B) MANC tag
ir_2_04B_1_ids = [15811, 16757, 17572, 18785, 166421]
ir_2_04B_2_ids = [17678, 17935, 18605, 18831, 21499, 21808, 22163, 26941, 29988, 100339, 100525, 153878, 164146, 165318]
ir_2_04B_3_ids = [20077,  101453] #cranial nuclei, ,primary neurite similar to 4 but braches  gou along anterior neuropil

ir_2_04B_4_ids =[20932, 21567, 36722, 38071, 158391,102138] #posterior nuclei,primary neurite similar to 3 but braches  gou along caudal neuropil

#5-7 are few neurons, different morphology than the rest; put them in separate clusters asthey aslo differ from each other. See ifthey math other subclasses
ir_2_04B_5_ids = [18641, 20836]
ir_2_04B_6_ids = [14389]
ir_2_04B_7_ids =[22997]
ir_remaining = [x for x in ir_04B if x not in ir_2_04B_1_ids and x not in ir_2_04B_2_ids and x not in ir_2_04B_4_ids and x not in ir_2_04B_5_ids]

#print(ir_remaining)



#did I miss anything: 
all_groups = (
    set(ir_2_04B_1_ids)
    | set(ir_2_04B_2_ids)
    | set(ir_2_04B_3_ids)
    | set(ir_2_04B_4_ids)
    | set(ir_2_04B_5_ids)
    | set(ir_2_04B_6_ids)
    | set(ir_2_04B_7_ids)
)


ir_set = set(ir_04B)

missing = ir_set - all_groups
extra = all_groups - ir_set

print("Missing from groups:", missing)
print("Extra in groups:", extra)


#did I duplicate any neuron?
from collections import Counter

all_ids_list = (
    ir_2_04B_1_ids
    + ir_2_04B_2_ids
    + ir_2_04B_3_ids
    + ir_2_04B_4_ids
    + ir_2_04B_5_ids
    + ir_2_04B_6_ids
    + ir_2_04B_8_ids
)

counts = Counter(all_ids_list)

duplicates = {k: v for k, v in counts.items() if v > 1}
print("Duplicates across groups:", duplicates)



Missing from groups: set()
Extra in groups: set()
Duplicates across groups: {}


Current view- manually generated 
https://clio-ng.janelia.org/#!%7B%22title%22:%22MANC_v1.2.3_4B_morphology_clusters_JHS%22%2C%22dimensions%22:%7B%22x%22:%5B8e-9%2C%22m%22%5D%2C%22y%22:%5B8e-9%2C%22m%22%5D%2C%22z%22:%5B8e-9%2C%22m%22%5D%7D%2C%22position%22:%5B17412.5%2C33790.5%2C52224.5%5D%2C%22crossSectionScale%22:1%2C%22projectionOrientation%22:%5B-0.11003785580396652%2C-0.47623971104621887%2C-0.647937536239624%2C0.5841783285140991%5D%2C%22projectionScale%22:6525.690625112658%2C%22layers%22:%5B%7B%22type%22:%22image%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22source%22%2C%22name%22:%22EM%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22source%22%2C%22segments%22:%5B%2217490%22%5D%2C%22colorSeed%22:3764239544%2C%22segmentColors%22:%7B%2217490%22:%22#87ceeb%22%7D%2C%22name%22:%22MANC_v1.2.3_BA%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22source%22%2C%22segments%22:%5B%2211143%22%2C%2214031%22%2C%2215393%22%2C%2215401%22%2C%2215969%22%2C%2216293%22%5D%2C%22colorSeed%22:1703097865%2C%22segmentColors%22:%7B%2211143%22:%22#ffa500%22%2C%2214031%22:%22#ffa500%22%2C%2215393%22:%22#ffa500%22%2C%2215401%22:%22#ffa500%22%2C%2215969%22:%22#ffa500%22%2C%2216293%22:%22#ffa500%22%7D%2C%22name%22:%22MANC_v1.2.3_BI%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2210024%22%2C%22101447%22%2C%2212523%22%2C%2213455%22%2C%2213506%22%2C%2213535%22%2C%2213589%22%2C%2214728%22%2C%2214774%22%2C%2215002%22%2C%2215005%22%2C%22152244%22%2C%2215309%22%2C%22155236%22%2C%22162540%22%2C%2216487%22%2C%2216527%22%2C%2217351%22%2C%2217409%22%2C%2217534%22%2C%2218148%22%2C%2218571%22%2C%2218755%22%2C%2219250%22%2C%2219366%22%2C%2221956%22%5D%2C%22colorSeed%22:2874648874%2C%22segmentDefaultColor%22:%22#ff0000%22%2C%22segmentColors%22:%7B%2210024%22:%22#e34234%22%2C%2212523%22:%22#e34234%22%2C%2213455%22:%22#e34234%22%2C%2213506%22:%22#e34234%22%2C%2213589%22:%22#e34234%22%2C%2214728%22:%22#e34234%22%2C%2214774%22:%22#e34234%22%2C%2215002%22:%22#e34234%22%2C%2215005%22:%22#e34234%22%2C%2215309%22:%22#e34234%22%2C%2216487%22:%22#e34234%22%2C%2216527%22:%22#e34234%22%2C%2217351%22:%22#e34234%22%2C%2217409%22:%22#e34234%22%2C%2217534%22:%22#e34234%22%2C%2218148%22:%22#e34234%22%2C%2218571%22:%22#e34234%22%2C%2218755%22:%22#e34234%22%2C%2219250%22:%22#e34234%22%2C%2219366%22:%22#e34234%22%2C%2221956%22:%22#e34234%22%2C%22101447%22:%22#e34234%22%2C%22152244%22:%22#e34234%22%2C%22155236%22:%22#e34234%22%2C%22162540%22:%22#e34234%22%2C%22164102%22:%22#e34234%22%7D%2C%22name%22:%22MANC_v1.2.3_BR%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22source%22%2C%22segments%22:%5B%2210314%22%2C%2213436%22%2C%2217395%22%2C%2219912%22%5D%2C%22colorSeed%22:1412771697%2C%22segmentColors%22:%7B%2210314%22:%22#cc99ff%22%2C%2213436%22:%22#cc99ff%22%2C%2217395%22:%22#cc99ff%22%2C%2219912%22:%22#cc99ff%22%7D%2C%22name%22:%22MANC_v1.2.3_IA%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22100339%22%2C%22100525%22%2C%22101453%22%2C%22102138%22%2C%22102536%22%2C%2213128%22%2C%2213409%22%2C%2213770%22%2C%2213798%22%2C%2213975%22%2C%2214024%22%2C%2214389%22%2C%2214697%22%2C%2214954%22%2C%22152655%22%2C%22153878%22%2C%2215811%22%2C%22158391%22%2C%2216224%22%2C%22164146%22%2C%22165318%22%2C%22166421%22%2C%2216757%22%2C%2217567%22%2C%2217572%22%2C%2217678%22%2C%2217935%22%2C%2218156%22%2C%2218253%22%2C%2218605%22%2C%2218629%22%2C%2218641%22%2C%2218724%22%2C%2218785%22%2C%2218831%22%2C%2218990%22%2C%2219833%22%2C%2219985%22%2C%2220077%22%2C%2220585%22%2C%2220836%22%2C%2220932%22%2C%2221189%22%2C%2221322%22%2C%2221499%22%2C%2221567%22%2C%2221808%22%2C%2221862%22%2C%2222163%22%2C%2222868%22%2C%2222997%22%2C%2223716%22%2C%2223790%22%2C%2224181%22%2C%2226388%22%2C%2226941%22%2C%2227760%22%2C%2229988%22%2C%2236722%22%2C%2238071%22%5D%2C%22colorSeed%22:2856349211%2C%22segmentColors%22:%7B%2213128%22:%22#66cdaa%22%2C%2213409%22:%22#66cdaa%22%2C%2213770%22:%22#66cdaa%22%2C%2213798%22:%22#66cdaa%22%2C%2213975%22:%22#66cdaa%22%2C%2214024%22:%22#66cdaa%22%2C%2214389%22:%22#66cdaa%22%2C%2214697%22:%22#66cdaa%22%2C%2214954%22:%22#66cdaa%22%2C%2215811%22:%22#66cdaa%22%2C%2216224%22:%22#66cdaa%22%2C%2216757%22:%22#66cdaa%22%2C%2217567%22:%22#66cdaa%22%2C%2217572%22:%22#66cdaa%22%2C%2217678%22:%22#66cdaa%22%2C%2217935%22:%22#66cdaa%22%2C%2218156%22:%22#66cdaa%22%2C%2218253%22:%22#66cdaa%22%2C%2218605%22:%22#66cdaa%22%2C%2218629%22:%22#66cdaa%22%2C%2218641%22:%22#66cdaa%22%2C%2218724%22:%22#66cdaa%22%2C%2218785%22:%22#66cdaa%22%2C%2218831%22:%22#66cdaa%22%2C%2218990%22:%22#66cdaa%22%2C%2219833%22:%22#66cdaa%22%2C%2219985%22:%22#66cdaa%22%2C%2220077%22:%22#66cdaa%22%2C%2220585%22:%22#66cdaa%22%2C%2220836%22:%22#66cdaa%22%2C%2220932%22:%22#66cdaa%22%2C%2221189%22:%22#66cdaa%22%2C%2221322%22:%22#66cdaa%22%2C%2221499%22:%22#66cdaa%22%2C%2221567%22:%22#66cdaa%22%2C%2221808%22:%22#66cdaa%22%2C%2221862%22:%22#66cdaa%22%2C%2222163%22:%22#66cdaa%22%2C%2222868%22:%22#66cdaa%22%2C%2222997%22:%22#66cdaa%22%2C%2223716%22:%22#66cdaa%22%2C%2223790%22:%22#66cdaa%22%2C%2224181%22:%22#66cdaa%22%2C%2226388%22:%22#66cdaa%22%2C%2226941%22:%22#66cdaa%22%2C%2227760%22:%22#66cdaa%22%2C%2229988%22:%22#66cdaa%22%2C%2236722%22:%22#66cdaa%22%2C%2238071%22:%22#66cdaa%22%2C%22100339%22:%22#66cdaa%22%2C%22100525%22:%22#66cdaa%22%2C%22101453%22:%22#66cdaa%22%2C%22102138%22:%22#66cdaa%22%2C%22102536%22:%22#66cdaa%22%2C%22152655%22:%22#66cdaa%22%2C%22153878%22:%22#66cdaa%22%2C%22158391%22:%22#66cdaa%22%2C%22164146%22:%22#66cdaa%22%2C%22165318%22:%22#66cdaa%22%2C%22166421%22:%22#66cdaa%22%7D%2C%22name%22:%22MANC_v1.2.3_IR%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2213506%22%5D%2C%22segmentQuery%22:%2213506%2C%2014728%2C%2014774%2C%2015005%2C%2018148%2C%2019250%2C%2019366%22%2C%22name%22:%22BR-2%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22source%22%2C%22segments%22:%5B%2210024%22%2C%22101447%22%2C%2212523%22%2C%22152244%22%2C%2215309%22%2C%22162540%22%2C%2216487%22%2C%2216527%22%2C%2217351%22%2C%2217409%22%2C%2217534%22%2C%2218571%22%2C%2221956%22%5D%2C%22segmentQuery%22:%2210024%2012523%2015309%2016487%2016527%2017351%2017409%2017534%20%2018571%2021956%20101447%20152244%20162540%22%2C%22name%22:%22BR-1%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22source%22%2C%22segments%22:%5B%2213455%22%2C%2215002%22%2C%22155236%22%2C%2218755%22%5D%2C%22segmentQuery%22:%2213455%2C%2013535%2C%2015002%2C%2018755%2C%20155236%22%2C%22name%22:%22BR-3%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22bounds%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22source%22%2C%22segments%22:%5B%22102536%22%2C%2213128%22%2C%2213409%22%2C%2213770%22%2C%2213798%22%2C%2213975%22%2C%2214024%22%2C%2214697%22%2C%2214954%22%2C%22152655%22%2C%2216224%22%2C%2217567%22%2C%2218156%22%2C%2218253%22%2C%2218629%22%2C%2218724%22%2C%2218990%22%2C%2219833%22%2C%2219985%22%2C%2220585%22%2C%2221189%22%2C%2221322%22%2C%2221862%22%2C%2222868%22%2C%2223716%22%2C%2223790%22%2C%2224181%22%2C%2226388%22%2C%2227760%22%5D%2C%22segmentQuery%22:%2213128%2C%2013409%2C%2013770%2C%2013798%2C%2013975%2C%2014024%2C%2014697%2C%2014954%2C%2016224%2C%2017567%2C%2018156%2C%2018253%2C%2018629%2C%2018724%2C%2018990%2C%2019833%2C%2019985%2C%2020585%2C%2021189%2C%2021322%2C%2021862%2C%2022868%2C%2023716%2C%2023790%2C%2024181%2C%2026388%2C%2027760%2C%20102536%2C%20152655%22%2C%22segmentDefaultColor%22:%22#ff0000%22%2C%22name%22:%22IR-1-20A?%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2215811%22%2C%22166421%22%2C%2216757%22%2C%2217572%22%2C%2218785%22%5D%2C%22segmentQuery%22:%2214389%2C%2015811%2C%2016757%2C%2017572%2C%2017678%2C%2017935%2C%2018605%2C%2018641%2C%2018785%2C%2018831%2C%2020077%2C%2020836%2C%2020932%2C%2021499%2C%2021567%2C%2021808%2C%2022163%2C%2022997%2C%2026941%2C%2029988%2C%2036722%2C%2038071%2C%20100339%2C%20101453%2C%20102138%2C%20153878%2C%20158391%2C%20164146%2C%20165318%2C%20166421%2C100525%22%2C%22name%22:%22IR-2-04B%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22source%22%2C%22segmentQuery%22:%2215811%2C%2016757%2C%2017572%2C%2018785%2C%20166421%22%2C%22name%22:%22IR-2-04B-1%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22100339%22%2C%22100525%22%2C%22153878%22%2C%22164146%22%2C%22165318%22%2C%2217678%22%2C%2217935%22%2C%2218605%22%2C%2218831%22%2C%2221499%22%2C%2221808%22%2C%2222163%22%2C%2226941%22%2C%2229988%22%5D%2C%22segmentQuery%22:%2217678%2C%2017935%2C%2018605%2C%2018831%2C%2021499%2C%2021808%2C%2022163%2C%2026941%2C%2029988%2C%20100339%2C%20100525%2C%20153878%2C%20164146%2C%20165318%22%2C%22name%22:%22IR-2-04B-2%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22101453%22%2C%2220077%22%5D%2C%22segmentQuery%22:%22%2020077%2C%20%20101453%2C%20%22%2C%22name%22:%22IR-2-04B-3%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2214389%22%5D%2C%22segmentQuery%22:%2214389%22%2C%22name%22:%22IR-2-04B-4%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%22158391%22%2C%2220932%22%2C%2221567%22%2C%2236722%22%2C%2238071%22%5D%2C%22segmentQuery%22:%2220932%2C%2021567%2C%2036722%2C%2038071%2C%20158391%22%2C%22name%22:%22IR-2-04B-5%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2222997%22%5D%2C%22segmentQuery%22:%2222997%22%2C%22name%22:%22IR-2-04B-6%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2218641%22%2C%2220836%22%5D%2C%22segmentQuery%22:%2218641%2C%2020836%22%2C%22name%22:%22IR-2-04B-7%22%7D%5D%2C%22selectedLayer%22:%7B%22flex%22:1.73%2C%22size%22:383%2C%22visible%22:true%2C%22layer%22:%22IR-2-04B-7%22%7D%2C%22crossSectionBackgroundColor%22:%22#ffffff%22%2C%22projectionBackgroundColor%22:%22#ffffff%22%2C%22layout%22:%223d%22%2C%22selection%22:%7B%22flex%22:0.27%2C%22size%22:383%7D%2C%22layerListPanel%22:%7B%22visible%22:true%7D%7D